# SW22–TW48 lead-time analysis notebook

This notebook documents the lead-time analysis for the rolling-window DEMETER experiment using the **SW22 / TW48** configuration. The main quantity calculated here is:

\[
\Delta t = t_{eq} - t_n
\]

where \(t_n\) is the anomaly time-window timestamp and \(t_{eq}\) is the first matched earthquake time listed for that anomaly. The goal is to compare the lead-time distribution of **model-selected seismic anomalies** against a **random/control anomaly sample**.

Main workflow:

1. Define output locations and helper functions.
2. Extract the first matched earthquake time from `matched_eqs`.
3. Generate random/control lead-time samples from `RDM_df_*` files.
4. Extract model anomaly lead times using the saved `seismic_indices`.
5. Compare random/control and model lead-time distributions.
6. Fit constant, linear, and exponential models to the binned lead-time counts.
7. Produce thesis-quality comparison plots.


## 1. Define the plot-output directory

This cell sets the output directory for SW22–TW48 lead-time plots. The path is Windows-specific and assumes the local project structure under `C:\PROJECT-DEMETER`.

In [ ]:
output_dir = fr'C:\PROJECT-DEMETER\Demeter_Retrain\output_base\Bg_window_data\New_BG\Results_Plot\SW22_TW48'

## 2. Helper function and one-window random-sampling check

This cell defines `extract_first_eq_time()`, which parses the `matched_eqs` field and returns the first earthquake time listed for an anomaly.

It then performs a one-window test using `summary_df_train_30D-22SW-tw48_w0.csv`:

- keeps only rows with `label == 1`;
- randomly samples anomaly rows;
- extracts the first matched earthquake time;
- computes the lead time in hours as \(\Delta t = t_{eq} - t_n\);
- prepares a `_Random100.csv` output table.

**Code note:** the loop uses `range(100)`, but the save command is commented out and the print message says “200 random samples”. If this cell is used again, align the loop count, filename, and print message.

In [ ]:
import pandas as pd
import numpy as np
import ast
import re
import os

def extract_first_eq_time(matched_eqs_str):
    """Extract the first listed earthquake time from matched_eqs string."""
    if not isinstance(matched_eqs_str, str) or matched_eqs_str.strip() in ["[]", "nan", "None"]:
        return pd.NaT

    # Replace Timestamp('...') with just '...'
    cleaned = re.sub(r"Timestamp\(['\"]?([^'\"]+)['\"]?\)", r"'\1'", matched_eqs_str)

    try:
        eq_list = ast.literal_eval(cleaned)
    except Exception:
        return pd.NaT

    if not eq_list:
        return pd.NaT

    # Take the FIRST element's Time
    first = eq_list[0].get("Time", None)
    if first:
        try:
            return pd.to_datetime(first)
        except Exception:
            return pd.NaT
    return pd.NaT




fname = r"C:\PROJECT-DEMETER\Demeter_Retrain\output_base\Bg_window_data\summary_df_train_30D-22SW-tw48_w0.csv"

df = pd.read_csv(fname)

# Only keep anomalies
df_anoms = df[df["label"] == 1].copy()

results = []

for i in range(100):   # repeat 200 times
    # randomly choose one anomaly row
    row = df_anoms.sample(1, random_state=np.random.randint(0, 999999)).iloc[0]
    
    # extract first EQ time
    first_eq_time = extract_first_eq_time(row["matched_eqs"])
    
    anomaly_time = pd.to_datetime(row["time_window"])
    delta_t = (first_eq_time - anomaly_time).total_seconds()/3600 if pd.notna(first_eq_time) else np.nan
    
    results.append({
        "index": row["index"],
        "time_window": anomaly_time,
        "label": row["label"],
        "matched_eqs": row["matched_eqs"],
        "total_eq": row["total_eq"],
        "first_eq_time": first_eq_time,
        "fname": os.path.basename(fname),
        "delta_t_hours": delta_t
    })

df_out = pd.DataFrame(results)
out_csv = fname.replace(".csv", "_Random100.csv")
# df_out.to_csv(out_csv, index=False)

print(f"✅ Saved 200 random samples from {fname} → {out_csv}")


## 3. Extract lead times from one model result

This section extracts model-selected seismic anomalies for one selected hyperparameter file, here `Retrain_result_Hp_B-tw48-30dBG_C.csv`.

For each train/validation rolling window, it:

1. reads the saved `seismic_indices`;
2. loads the corresponding detailed `summary_df_*` file;
3. selects rows whose `index` appears in `seismic_indices`;
4. extracts the first matched earthquake time;
5. computes the lead time \(\Delta t\) in hours;
6. saves one `SeismicAnomalies_<split>_w<window>_22sw_48w.csv` file per split and window.

This is the first model-based lead-time extraction step.

In [ ]:
import pandas as pd
import numpy as np
import os
import ast
import re

# ---- Helper to parse matched_eqs and extract first EQ time ----
# ---- Paths ----
base_path = r"C:\PROJECT-DEMETER\Demeter_Retrain\output_base\Bg_window_data"
# summary_file = os.path.join(base_path, "New_BG", "Results_Plot", "result_Hp_eq_tw48-30dBG_C.csv")
summary_file = os.path.join(base_path, "New_BG", "Results_csv",'SW22_TW48', "Retrain_result_Hp_B-tw48-30dBG_C.csv")

# ---- Load summary (with seismic_indices) ----
k = pd.read_csv(summary_file)

all_results = []

# ---- Iterate rows in results ----
for idx, row in k.iterrows():
    split = row["split"]
    window = idx // 2  # derive window number
    fname = os.path.join(base_path, f"summary_df_{split.lower()}_30D-22SW-tw48_w{window}.csv")
    
    if not os.path.exists(fname):
        print(f"⚠️ Missing file: {fname}")
        continue
    
    # Load per-window data
    df = pd.read_csv(fname, parse_dates=["time_window"])
    
    # Parse seismic indices (string of list)
    try:
        seismic_indices = ast.literal_eval(str(row["seismic_indices"]))
    except Exception:
        seismic_indices = []
    
    df_sel = df[df["index"].isin(seismic_indices)].copy()
    
    # Compute first_eq_time and delta_t
    records = []
    for _, r in df_sel.iterrows():
        first_eq_time = extract_first_eq_time(r["matched_eqs"])
        anomaly_time = pd.to_datetime(r["time_window"])
        delta_t = (first_eq_time - anomaly_time).total_seconds()/3600 if pd.notna(first_eq_time) else np.nan
        
        records.append({
            "index": r["index"],
            "time_window": anomaly_time,
            "label": r["label"] if "label" in r else r.get("Label", None),
            "matched_eqs": r["matched_eqs"],
            "total_eq": r["total_eq"],
            "first_eq_time": first_eq_time,
            "delta_t_hours": delta_t,
            "split": split,
            "window": window,
            "fname": os.path.basename(fname)
        })
    
    df_out = pd.DataFrame(records)
    
    # Save CSV for this split+window
    out_name = f"SeismicAnomalies_{split}_w{window}_22sw_48w.csv"
    out_path = os.path.join(base_path, "New_BG", "Results_csv", out_name)
    df_out.to_csv(out_path, index=False)
    print(f"✅ Saved {out_path}")
    
    all_results.append(df_out)

# ---- Combine all into one file ----
# final_df = pd.concat(all_results, ignore_index=True)
# final_path = os.path.join(base_path, "New_BG", "Results_Plot", "All_SeismicAnomalies.csv")
# final_df.to_csv(final_path, index=False)
# print(f"✅ Combined file saved: {final_path}")


## 4. Generate random/control lead-time samples for validation windows

This cell processes validation-window random/control files:

`RDM_df_val_30D-22SW-tw48_w<window>.csv`

For each rolling window \(w = 0,\ldots,15\), it keeps only rows with `label == 1`, samples rows at random, extracts the first matched earthquake, and stores \(\Delta t\) in hours.

**Important code note:** the loop uses `range(100)`, so it creates 100 sampled rows per window, although the output filename uses `_Random100.csv` and the print message says 100. If the intended design is 100 random draws per window, change `range(100)` to `range(100)`.

Developing Time randomised sequece labels

In [ ]:
import pandas as pd
import random

def label_sequences_RDM(dataset, earthquakes, spatial_width, time_window_hours, save_path=None):
    """
    Label each half-orbit sequence based on whether any location within the sequence
    is seismic within a configured spatial/time window, using a random offset (5–30 days)
    from the last timestamp.

    Args:
        dataset: HalfOrbitPairDataset instance (e.g., train/val/test dataset).
        earthquakes (pd.DataFrame): Earthquake catalog with ['Time','lat','long','mag'].
        spatial_width (float): Spatial proximity threshold (e.g., 20 degrees).
        time_window_hours (float): Time window threshold (in hours).
        save_path (str, optional): If provided, CSV path to save the summary.

    Returns:
        pd.DataFrame with columns:
        ['index', 'time_window', 'label', 'matched_eqs', 'total_eq', 'random_offset_days']
    """

    # Create a criteria instance for seismic proximity checking
    criteria = SeismicCriteria(spatial_width, time_window_hours)

    # Build sequences (from dataset.df)
    _, dt_seqs, latlon_seqs, _ = dataset.create_half_orbit_sequences(dataset.df)

    sequence_summary = []
    matched_eqs_records = []  

    for idx, (dt_seq, latlon_seq) in enumerate(zip(dt_seqs, latlon_seqs)):
        # Random offset between 5 and 30 days
        random_days = random.randint(5, 30)
        anomaly_time = dt_seq[-1] + pd.Timedelta(days=random_days)

        label = 0
        seq_matched_eqs = []

        # Check seismic condition for all locations in sequence
        for loc in latlon_seq:
            is_seismic, inside_eqs, _ = criteria.is_eq(anomaly_time, loc, earthquakes)
            if inside_eqs:
                seq_matched_eqs.extend(inside_eqs)
            if is_seismic:
                label = 1

        sequence_summary.append({
            "index": idx,
            "time_window": anomaly_time,
            "label": label,
            "matched_eqs": seq_matched_eqs,
            "total_eq": len(seq_matched_eqs),
            "random_offset_days": random_days
        })

        matched_eqs_records.extend(seq_matched_eqs)

    summary_df = pd.DataFrame(sequence_summary).reset_index(drop=True)

    # Unique EQs matched
    if matched_eqs_records:
        matched_eqs_df = pd.DataFrame(matched_eqs_records).drop_duplicates()
    else:
        matched_eqs_df = pd.DataFrame(columns=["lat", "long", "Time", "mag"])

    # Optional: save results
    if save_path:
        summary_df.to_csv(save_path, index=False)
        print(f"[INFO] Saved summary DataFrame to {save_path}")

    print(f"[INFO] Total labeled sequences: {len(summary_df)}")
    print(f"[INFO] Unique earthquakes matched: {len(matched_eqs_df)}")

    return summary_df

# Example: generate windows
windows_train = generate_windows(start_date="2005-01-01 00:00:00", end_date="2011-01-02 00:00:00",
                           train_months=12, val_months=3)

all_results = []
# iterate through windows
for i, w in enumerate(windows_train):
      # if i== 0:

      tag_l = f'tw{tw}_w{i}'


      print(f"Window {i}: Train {w['train_start']} → {w['train_end']}, "
            f"Val {w['val_start']} → {w['val_end']}")
      df1 =  pd.read_pickle(f"C:\PROJECT-DEMETER\Demeter_Retrain\output_base\Bg_window_data\Background_data-window_{i}.pkl")
      # subset data
      train_set = df1[(df1.index >= w['train_start']) & (df1.index < w['train_end'])]
      val_set  = df1[(df1.index >= w['val_start'])   & (df1.index < w['val_end'])]
      test_set  = pd.DataFrame(df[df.index >= '2010-01-01'])

      # datasets (non-seismic only for fitting scaler + training)
      train_dataset = HalfOrbitPairDataset(train_set, min_data_points=min_data_points)
      val_dataset   = HalfOrbitPairDataset(val_set,   min_data_points=min_data_points)

      seismic_criteria = SeismicCriteria(spatial_width=sw, time_window_hours=tw)
            
      # label_sequences(dataset, earthquakes, spatial_width, time_window_hours, save_path=None)
      df_train_labels = label_sequences_RDM(dataset=train_dataset, earthquakes=eq,spatial_width=22, time_window_hours=tw,
                                                            save_path=os.path.join(output_dir, f"RDM_df_train_30D-{sw}SW-{tag_l}.csv"))
      df_val_labels   =label_sequences_RDM(dataset=val_dataset,   earthquakes=eq,spatial_width=sw, time_window_hours=tw,
                                                            save_path=os.path.join(output_dir, f"RDM_df_val_30D-{sw}SW-{tag_l}.csv"))

    

    


In [ ]:
windows  = np.arange(0,16) 
for w in windows:
    # df_train  = pd.read_csv(f"C:\PROJECT-DEMETER\Demeter_Retrain\output_base\Bg_window_data\summary_df_train_30D-tw48_w{w}.csv")


    fname = fr"C:\PROJECT-DEMETER\Demeter_Retrain\output_base\Bg_window_data\RDM_df_val_30D-22SW-tw48_w{w}.csv"

    df_anoms = pd.read_csv(fname)

    # Only keep anomalies
    df_anoms = df_anoms[df_anoms["label"] == 1].copy()

    results = []

    for i in range(100):   # repeat 200 times
        # randomly choose one anomaly row
        row = df_anoms.sample(1, random_state=np.random.randint(0, 999999)).iloc[0]
        
        # extract first EQ time
        first_eq_time = extract_first_eq_time(row["matched_eqs"])
        
        anomaly_time = pd.to_datetime(row["time_window"])
        delta_t = (first_eq_time - anomaly_time).total_seconds()/3600 if pd.notna(first_eq_time) else np.nan
        
        results.append({
            "index": row["index"],
            "time_window": anomaly_time,
            "label": row["label"],
            "matched_eqs": row["matched_eqs"],
            "total_eq": row["total_eq"],
            "first_eq_time": first_eq_time,
            "fname": os.path.basename(fname),
            "delta_t_hours": delta_t
        })

    df_out = pd.DataFrame(results)
    out_csv = fname.replace(".csv", "_Random100.csv")
    df_out.to_csv(out_csv, index=False)

    print(f"✅ Saved 100 random samples from {fname} → {out_csv}")


## 5. Generate random/control lead-time samples for training windows

This is the same random/control sampling procedure as above, but applied to training-window files:

`RDM_df_train_30D-22SW-tw48_w<window>.csv`

For each window, the code keeps `label == 1` rows, repeatedly samples one row, extracts the first matched earthquake time, computes \(\Delta t\), and writes a `_Random100.csv` file.

**Important code note:** this cell also uses `range(100)`, so it creates 100 rows per window despite the `_Random100.csv` filename and the “200 random samples” print message.

In [ ]:
windows  = np.arange(0,16) 
for w in windows:
    # df_train  = pd.read_csv(f"C:\PROJECT-DEMETER\Demeter_Retrain\output_base\Bg_window_data\summary_df_train_30D-tw48_w{w}.csv")


    fname = fr"C:\PROJECT-DEMETER\Demeter_Retrain\output_base\Bg_window_data\RDM_df_train_30D-22SW-tw48_w{w}.csv"

    df_anoms = pd.read_csv(fname)

    # Only keep anomalies
    df_anoms = df_anoms[df_anoms["label"] == 1].copy()

    results = []

    for i in range(100):   # repeat 200 times
        # randomly choose one anomaly row
        row = df_anoms.sample(1, random_state=np.random.randint(0, 999999)).iloc[0]
        # print(row)
        
        # extract first EQ time
        first_eq_time = extract_first_eq_time(row["matched_eqs"])
        
        anomaly_time = pd.to_datetime(row["time_window"])
        delta_t = (first_eq_time - anomaly_time).total_seconds()/3600 if pd.notna(first_eq_time) else np.nan
        
        results.append({
            "index": row["index"],
            "time_window": anomaly_time,
            "label": row["label"],
            "matched_eqs": row["matched_eqs"],
            "total_eq": row["total_eq"],
            "first_eq_time": first_eq_time,
            "fname": os.path.basename(fname),
            "delta_t_hours": delta_t
        })

    df_out = pd.DataFrame(results)
    out_csv = fname.replace(".csv", "_Random100.csv")
    df_out.to_csv(out_csv, index=False)

    print(f"✅ Saved 200 random samples from {fname} → {out_csv}")


## 6. Combine random/control samples and plot their overall lead-time distribution

This cell loads all previously saved `_Random100.csv` files, concatenates them into one dataframe, and plots the overall distribution of random/control lead times.

The histogram shows the count distribution of \(\Delta t = t_{eq} - t_n\) in hours across all sampled random/control anomaly rows.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import glob
import os

# Path where your *_Random200.csv files are
base_path = r"C:\PROJECT-DEMETER\Demeter_Retrain\output_base\Bg_window_data"

# Load all Random200 files
all_files = glob.glob(os.path.join(base_path, "*30D-22SW-tw48_w*_Random100.csv"))

df_list = []
for f in all_files:
    df = pd.read_csv(f, parse_dates=["time_window","first_eq_time"])
    df["fname"] = os.path.basename(f)
    df_list.append(df)

df_all = pd.concat(df_list, ignore_index=True)

# Δt values (hours → days maybe clearer?)
dt = df_all["delta_t_hours"].dropna()  # convert to days

# Plot histogram → dn/dΔt vs Δt
plt.figure(figsize=(8,6))
counts, bins, patches = plt.hist(dt, bins=20, density=False, alpha=0.7, edgecolor="k")

plt.xlabel(r"$t_{eq} - t_n$  (hours)")
plt.ylabel(r'counts')
plt.title("Distribution of Random anomaly–earthquake time differences across all windows")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 7. Compare random and model lead-time histograms for train and validation splits

This section loads:

- random samples from `RDM_df_<split>_30D-22SW-tw48_w*_Random100.csv`;
- model anomaly files from `SeismicAnomalies_<split>_w*_22sw_48w.csv`.

It then plots overlaid histograms for train and validation splits. The purpose is to visually compare whether the model-selected anomaly lead times show a different temporal structure from the random/control samples.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import glob
import os
import re

base_path = r"C:\PROJECT-DEMETER\Demeter_Retrain\output_base\Bg_window_data"
res_path  = os.path.join(base_path, "New_BG", "Results_csv")  # where SeismicAnomalies_* are saved


def load_random(split):
    """Load random anomaly files for given split (Train/Val) with w <= 15."""
    files = glob.glob(os.path.join(base_path, f"RDM_df_{split}_30D-22SW-tw48_w*_Random100.csv"))
    
    df_list = []
    for f in files:
        # extract window number from filename
        m = re.search(r"_w(\d+)_Random100\.csv", os.path.basename(f))
        if not m:
            continue
        
        w = int(m.group(1))
        
        # keep only windows <= 15
        if w > 15:
            continue
        
        df = pd.read_csv(f, parse_dates=["time_window", "first_eq_time"])
        df["window"] = w
        df["split"] = split
        df_list.append(df)
    
    if not df_list:
        # return empty DataFrame with expected columns if nothing found
        return pd.DataFrame(columns=["time_window", "first_eq_time", "window", "split"])
    
    return pd.concat(df_list, ignore_index=True)

def load_model(split):
    """Load model anomaly results for given split (train/val)."""
    files = glob.glob(os.path.join(base_path, "New_BG", "Results_csv",  f"SeismicAnomalies_{split}_w*_22sw_48w.csv"))
    print(f"[{split}] Found {len(files)} files in {res_path}")

    df_list = []
    for f in files:
        m = re.search(r"_w(\d+).*\.csv", os.path.basename(f))
        if not m:
            print(f"⚠️ Could not extract window number from: {f}")
            continue
        w = int(m.group(1))
        df = pd.read_csv(f, parse_dates=["time_window", "first_eq_time"])
        df["window"] = w
        df["split"] = split
        df_list.append(df)

    if not df_list:
        raise ValueError(f"No valid SeismicAnomalies files found for split={split}")

    return pd.concat(df_list, ignore_index=True)


# Load both sets
df_rand_train = load_random("Train")
df_rand_val   = load_random("Val")
df_model_train = load_model("Train")
df_model_val   = load_model("Val")

# --- Plot Train ---
plt.figure(figsize=(8,6))
rand_dt = df_rand_train["delta_t_hours"].dropna()
model_dt = df_model_train["delta_t_hours"].dropna()
bins = np.arange(min(rand_dt.min(), model_dt.min()),
                 max(rand_dt.max(), model_dt.max())+2, 1)  # 1-hour bins
plt.hist(rand_dt, bins=bins, alpha=0.5, color="blue", label="Random Train", edgecolor="k")
plt.hist(model_dt, bins=bins, alpha=0.5, color="red",  label="Model Train", edgecolor="k")
plt.xlabel("Δt (hours)")
plt.ylabel("Count")
plt.title("Δt Distribution (Train windows)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# --- Plot Val ---
plt.figure(figsize=(8,6))
rand_dt = df_rand_val["delta_t_hours"].dropna()
model_dt = df_model_val["delta_t_hours"].dropna()
bins = np.arange(min(rand_dt.min(), model_dt.min()),
                 max(rand_dt.max(), model_dt.max())+2, 1)
plt.hist(rand_dt, bins=bins, alpha=0.5, color="blue", label="Random Val", edgecolor="k")
plt.hist(model_dt, bins=bins, alpha=0.5, color="red",  label="Model Val", edgecolor="k")
plt.xlabel("Δt (hours)")
plt.ylabel("Count")
plt.title("Δt Distribution (Validation windows)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 8. Extract lead times for all hyperparameter models

This cell generalizes the single-model extraction step to all listed hyperparameter configurations.

For each model, it reads the corresponding `Retrain_result_Hp_B-<model_name>.csv`, extracts the saved `seismic_indices`, loads the detailed per-window summary data, computes `first_eq_time` and `delta_t_hours`, attaches the model hyperparameters, and saves two combined output files:

- `ALL_MODELS_SeismicAnomalies_TRAIN.csv`
- `ALL_MODELS_SeismicAnomalies_VAL.csv`

These combined files are later used for model-wise lead-time comparison and plotting.

**Code note:** the notebook title and filenames indicate SW22/TW48, but this cell sets `sw = 48`. In this cell, `sw` appears to be stored only as metadata in `df_hparams`; it does not change the file paths used for the analysis. For reporting consistency, verify whether this should be `sw = 22`.

In [ ]:
import pandas as pd
import numpy as np
import os
import ast
import re
import warnings

tw = 48
sw = 48
# =============================================================================
# 2. MODEL HYPERPARAMETERS
# =============================================================================

import pandas as pd

data = [

    {"Naming": f"tw{tw}-30dBG_A", "Learning rate": 0.0001, "Hidden size": 8, "Number of Layers": 2, "Batch size": 8, "tw" :f"{tw}" , "sw" : f"{sw}"},
    
    {"Naming": f"tw{tw}-30dBG_B", "Learning rate": 0.0001, "Hidden size": 12, "Number of Layers": 2, "Batch size": 8, "tw"  :f"{tw}" , "sw" : f"{sw}" },

    {"Naming": f"tw{tw}-30dBG_C", "Learning rate": 0.0001, "Hidden size": 18, "Number of Layers": 2, "Batch size": 8, "tw"  :f"{tw}" , "sw" : f"{sw}" },

    {"Naming": f"tw{tw}-30dBG_D", "Learning rate": 0.0001, "Hidden size": 32, "Number of Layers": 2, "Batch size": 8, "tw"  :f"{tw}" , "sw" : f"{sw}" },

    {"Naming": f"tw{tw}-30dBG_E", "Learning rate": 0.0001, "Hidden size": 64, "Number of Layers": 2, "Batch size": 8, "tw"  :f"{tw}" , "sw" : f"{sw}"},

    {"Naming": f"tw{tw}-30dBG_In-A", "Learning rate": 0.0001, "Hidden size": 8, "Number of Layers": 2, "Batch size": 8, "tw" :f"{tw}" , "sw" : f"{sw}" },
    
    {"Naming": f"tw{tw}-30dBG_In-B", "Learning rate": 0.0001, "Hidden size": 12, "Number of Layers": 2, "Batch size": 8, "tw"  :f"{tw}" , "sw" : f"{sw}"},
    
    {"Naming": f"tw{tw}-30dBG_In-C", "Learning rate": 0.0001, "Hidden size": 18, "Number of Layers": 2, "Batch size": 8, "tw"  :f"{tw}" , "sw" : f"{sw}" },

    {"Naming": f"tw{tw}-30dBG_In-D", "Learning rate": 0.0001, "Hidden size": 32, "Number of Layers": 2, "Batch size": 8, "tw"  :f"{tw}" , "sw" : f"{sw}" },
    
    {"Naming": f"tw{tw}-30dBG_In-E", "Learning rate": 0.0001, "Hidden size": 64, "Number of Layers": 2, "Batch size": 8, "tw"  :f"{tw}" , "sw" : f"{sw}" },
    
]

df_hparams = pd.DataFrame(data)

# ---- Base Paths ----
base_path = r"C:\PROJECT-DEMETER\Demeter_Retrain\output_base\Bg_window_data"
results_csv_path = os.path.join(base_path, "New_BG", "Results_csv", "SW22_TW48")


# This will collect the final DataFrame from *each* model
all_models_results_list = []

# Suppress warnings from pd.to_datetime for invalid formats
warnings.filterwarnings("ignore", category=UserWarning, module='pandas')

# ---- MAIN MODEL LOOP ----
print("Starting processing for all models to get lead times...")
for _, model_row in df_hparams.iterrows():
    model_name = model_row["Naming"]
    
    summary_file = os.path.join(results_csv_path, f"Retrain_result_Hp_B-{model_name}.csv")
    
    if not os.path.exists(summary_file):
        print(f"⚠️ Missing summary file for model {model_name}. Skipping: {summary_file}")
        continue

    print(f"--- Processing Model: {model_name} ---")
    k = pd.read_csv(summary_file)
    
    # This list will hold all window-data for *this* model
    current_model_window_results = []

    # ---- Iterate rows/windows in this model's results ----
    for idx, row in k.iterrows():
        split = row["split"]
        # Derive window number. Assumes train/val rows are interleaved.
        window = idx // 2  
        
        # This is the path to the file with the DETAILED anomaly info
        fname = os.path.join(base_path, f"summary_df_{split.lower()}_30D-22SW-tw48_w{window}.csv")
        
        if not os.path.exists(fname):
            print(f"  -> ⚠️ Missing window file: {fname}")
            continue
        
        # Load per-window data
        try:
            df = pd.read_csv(fname, parse_dates=["time_window"])
        except Exception as e:
            print(f"  -> ⚠️ Error reading {fname}: {e}")
            continue
        
        # Parse seismic indices (string of list)
        try:
            seismic_indices = ast.literal_eval(str(row["seismic_indices"]))
        except Exception:
            seismic_indices = []
        
        if not seismic_indices:
            continue
            
        # Select only the rows that were flagged as seismic
        df_sel = df[df["index"].isin(seismic_indices)].copy()
        
        if df_sel.empty:
            continue
            
        # --- Compute first_eq_time and delta_t ---
        records = []
        for _, r in df_sel.iterrows():
            first_eq_time = extract_first_eq_time(r["matched_eqs"])
            anomaly_time = pd.to_datetime(r["time_window"])
            
            # This is the lead time calculation
            delta_t = (first_eq_time - anomaly_time).total_seconds()/3600 if pd.notna(first_eq_time) else np.nan
            
            # Create the record dictionary
            record_data = {
                "index": r["index"],
                "time_window": anomaly_time,
                "label": r["label"] if "label" in r else r.get("Label", None),
                "matched_eqs": r["matched_eqs"],
                "total_eq": r["total_eq"],
                "first_eq_time": first_eq_time,
                "delta_t_hours": delta_t,  # <-- LEAD TIME
                "split": split,
                "window": window,
                "model_name": model_name
            }
            
            # Add all other hyperparameters dynamically
            for col_name in df_hparams.columns.drop("Naming"):
                clean_col_name = f"model_{col_name.lower().replace(' ', '_')}"
                record_data[clean_col_name] = model_row[col_name]
            
            records.append(record_data)
        
        if records:
            df_out = pd.DataFrame(records)
            current_model_window_results.append(df_out)
    
    # After processing all windows for *this* model, combine them
    if current_model_window_results:
        model_final_df = pd.concat(current_model_window_results, ignore_index=True)
        all_models_results_list.append(model_final_df)
        print(f"  -> ✅ Processed {len(k)} windows, found {len(model_final_df)} seismic anomalies for {model_name}.")
    else:
        print(f"  -> ℹ️ No seismic anomalies found for {model_name}.")

# =============================================================================
# 4. FINAL COMBINATION AND SPLITTING (TRAIN / VAL)
# =============================================================================

if all_models_results_list:
    print("\n--- Combining all model results into one file ---")
    
    # Combine all results from all models into ONE DataFrame
    final_df = pd.concat(all_models_results_list, ignore_index=True)
    
    # --- SPLIT THE FINAL DATAFRAME ---
    final_train_df = final_df[final_df['split'].str.lower() == 'train'].copy()
    final_val_df = final_df[final_df['split'].str.lower() == 'val'].copy()

    # Define final output paths
    final_path_train = os.path.join(results_csv_path, "ALL_MODELS_SeismicAnomalies_TRAIN.csv")
    final_path_val = os.path.join(results_csv_path, "ALL_MODELS_SeismicAnomalies_VAL.csv")
    
    # Save the two separate files
    final_train_df.to_csv(final_path_train, index=False)
    final_val_df.to_csv(final_path_val, index=False)
    
    print("\n" + "="*50)
    print(f"✅✅✅ Combined TRAIN file saved: {final_path_train}")
    print(f"     Total train seismic anomalies: {len(final_train_df)}")
    print(f"✅✅✅ Combined VAL file saved: {final_path_val}")
    print(f"     Total val seismic anomalies: {len(final_val_df)}")
else:
    print("\n--- No results found for any models ---")

## 9. Load the combined model anomaly tables

This cell defines a loader for the combined model anomaly outputs created in the previous step.

The function `load_model(split)` reads either:

- `ALL_MODELS_SeismicAnomalies_TRAIN.csv`, or
- `ALL_MODELS_SeismicAnomalies_VAL.csv`.

These files already contain the computed `delta_t_hours`, so they can be used directly for histogramming and fitting.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import glob
import os
import re
import numpy as np

# --- Paths ---
# Base path to all data
base_path = r"C:\PROJECT-DEMETER\Demeter_Retrain\output_base\Bg_window_data"
# Path to the 'SW2022' folder where the FINAL combined files are saved
model_res_path = os.path.join(base_path, "New_BG", "Results_csv", 'SW22_TW48') 
# Where to save the plots


print(f"Base path set to: {base_path}")
print(f"Model Results path set to: {model_res_path}")



def load_model(split):
    """
    Load the SINGLE, COMBINED model anomaly file for the given split.
    This is much more reliable than loading many small files.
    """
    file_path = os.path.join(model_res_path, f"ALL_MODELS_SeismicAnomalies_{split.upper()}.csv")
    print(f"Loading single model file: {file_path}")

    if not os.path.exists(file_path):
        print(f"--- ERROR: File not found: {file_path} ---")
        print("--- Please run the previous data processing script first. ---")
        return pd.DataFrame() # Return empty DF

    try:
        # These CSVs already contain 'delta_t_hours'
        df = pd.read_csv(file_path, parse_dates=["time_window","first_eq_time"])
        print(f"Successfully loaded {file_path}, {len(df)} rows.")
        return df
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        return pd.DataFrame()


## 10. Define statistical fitting tools for lead-time histograms

This section defines the functions used to fit the binned lead-time distributions.

The fitted models are:

1. **Constant model**: checks whether counts are approximately flat across lead time.
2. **Linear model**: checks whether counts increase or decrease approximately linearly with lead time.
3. **Exponential model**: checks whether counts decay or grow exponentially with lead time.

The reduced chi-square statistic is used as a goodness-of-fit summary:

\[
\chi^2_{red} = \frac{1}{N-p}\sum_i \left(\frac{y_i - f(x_i)}{\sigma_i}\right)^2
\]

where \(N\) is the number of nonzero bins, \(p\) is the number of fitted parameters, and the uncertainty is approximated as \(\sigma_i = \sqrt{y_i}\), consistent with Poisson counting uncertainty.

The cell also reloads the random/control and combined model tables, then identifies all available model names.

In [ ]:
import numpy as np
from scipy.optimize import curve_fit
from scipy.stats import linregress

# ---------------------------
# Utility Functions
# ---------------------------

def reduced_chi_square(y_obs, y_exp, sigma, n_params):
    chi2 = np.sum(((y_obs - y_exp) / sigma) ** 2)
    dof = len(y_obs) - n_params
    return chi2 / dof if dof > 0 else np.nan


def exponential_model(x, a, b):
    return a * np.exp(-b * x)


# ---------------------------
# Fit function with uncertainties
# ---------------------------
def compute_fits(x, y, label):
    results = {}

    # Guard for no/zero data
    if len(y) == 0 or np.all(y <= 0):
        print(f"\n--- {label} ---")
        print("No positive counts to fit. Skipping.")
        return results

    # Filter zeros or negatives (avoid log(0) and zero sigma)
    mask = y > 0
    x_fit, y_fit = x[mask], y[mask]
    if len(y_fit) == 0:
        print(f"\n--- {label} ---")
        print("All counts are zero after filtering. Skipping.")
        return results

    sigma = np.sqrt(y_fit)

    # =====================================================
    # --- Constant fit ---
    # =====================================================
    const = np.mean(y_fit)
    const_y = np.full_like(y_fit, const)

    chi_const = reduced_chi_square(y_fit, const_y, sigma, 1)

    # standard error of mean
    const_err = np.std(y_fit, ddof=1) / np.sqrt(len(y_fit))

    results["constant"] = {
        "C": const,
        "C_err": const_err,
        "chi2_red": chi_const
    }

    # =====================================================
    # --- Linear fit ---
    # =====================================================
    slope, intercept, r_value, p_value, std_err = linregress(x_fit, y_fit)

    y_lin = slope * x_fit + intercept
    chi_lin = reduced_chi_square(y_fit, y_lin, sigma, 2)

    # intercept uncertainty
    n = len(x_fit)
    x_mean = np.mean(x_fit)
    # Correct intercept uncertainty
    residuals = y_fit - (slope * x_fit + intercept)
    s = np.sqrt(np.sum(residuals**2) / (n - 2))          # residual std dev
    Sxx = np.sum((x_fit - x_mean)**2)                    # sum of squared deviations

    slope_err_correct = s / np.sqrt(Sxx)                 # should match std_err from linregress
    intercept_err_correct = s * np.sqrt(np.sum(x_fit**2) / (n * Sxx))

    intercept_err = std_err * np.sqrt(
        np.sum(x_fit**2) / n
    )

    results["linear"] = {
        "a": slope,
        "a_err": slope_err_correct,
        "b": intercept,
        "b_err": intercept_err_correct,
        "chi2_red": chi_lin
    }

    # =====================================================
    # --- Exponential fit ---
    # =====================================================
    try:
        a0 = max(y_fit[0], 1e-3)
        b0 = 1 / (x_fit.max() - x_fit.min() + 1e-9)

        p0 = [a0, b0]

        print(f"\nInitial guesses for {label}: a0={a0:.3f}, b0={b0:.3f}")

        bounds = ([0, -np.inf], [np.inf, np.inf])

        popt, pcov = curve_fit(
            exponential_model,
            x_fit,
            y_fit,
            p0=p0,
            bounds=bounds,
            maxfev=1000
        )

        # parameter uncertainties
        perr = np.sqrt(np.diag(pcov))

        y_exp = exponential_model(x_fit, *popt)

        chi_exp = reduced_chi_square(
            y_fit,
            y_exp,
            sigma,
            len(popt)
        )

        results["exponential"] = {
            "A": popt[0],
            "A_err": perr[0],
            "B": popt[1],
            "B_err": perr[1],
            "chi2_red": chi_exp,
        }

    except Exception as e:
        print(f"Exponential fit failed for {label}: {e}")

        results["exponential"] = {
            "A": np.nan,
            "A_err": np.nan,
            "B": np.nan,
            "B_err": np.nan,
            "chi2_red": np.nan
        }

    # =====================================================
    # --- Print summary ---
    # =====================================================
    print(f"\n--- {label} ---")

    print(
        f"Constant fit : "
        f"C = {results['constant']['C']:.3f} ± {results['constant']['C_err']:.3f}, "
        f"χ²_red = {results['constant']['chi2_red']:.2f}"
    )

    print(
        f"Linear fit   : "
        f"a = {results['linear']['a']:.5f} ± {results['linear']['a_err']:.5f}, "
        f"b = {results['linear']['b']:.3f} ± {results['linear']['b_err']:.3f}, "
        f"χ²_red = {results['linear']['chi2_red']:.2f}"
    )

    print(
        f"Exponential fit : "
        f"A = {results['exponential']['A']:.3f} ± {results['exponential']['A_err']:.3f}, "
        f"B = {results['exponential']['B']:.6f} ± {results['exponential']['B_err']:.6f}, "
        f"χ²_red = {results['exponential']['chi2_red']:.2f}"
    )

    # Also return the x subset used for fitting
    results["_x_fit"] = x_fit

    return results
# def compute_fits_with_uncertainties(x, y, label):
#     results = {}

#     if len(y) == 0 or np.all(y <= 0):
#         print(f"\n--- {label} ---")
#         print("No positive counts to fit. Skipping.")
#         return results

#     mask = y > 0
#     x_fit, y_fit = x[mask], y[mask]

#     sigma = np.sqrt(y_fit)
#     sigma[sigma == 0] = 1.0

#     n = len(y_fit)

#     # =====================================================
#     # 1. Constant fit
#     # =====================================================
#     weights = 1 / sigma**2

#     C = np.sum(weights * y_fit) / np.sum(weights)
#     C_err = np.sqrt(1 / np.sum(weights))

#     y_const = np.full_like(y_fit, C, dtype=float)
#     chi_const = reduced_chi_square(y_fit, y_const, sigma, 1)

#     results["constant"] = {
#         "C": C,
#         "C_err": C_err,
#         "chi2_red": chi_const
#     }

#     # =====================================================
#     # 2. Linear fit: y = a*x + b
#     # Weighted linear least-squares
#     # =====================================================
#     coeffs, cov = np.polyfit(x_fit, y_fit, deg=1, w=1/sigma, cov=True)

#     a_lin = coeffs[0]
#     b_lin = coeffs[1]

#     a_lin_err = np.sqrt(cov[0, 0])
#     b_lin_err = np.sqrt(cov[1, 1])

#     y_lin = a_lin * x_fit + b_lin
#     chi_lin = reduced_chi_square(y_fit, y_lin, sigma, 2)

#     results["linear"] = {
#         "a": a_lin,
#         "a_err": a_lin_err,
#         "b": b_lin,
#         "b_err": b_lin_err,
#         "chi2_red": chi_lin
#     }

#     # =====================================================
#     # 3. Exponential fit: y = A exp(-B x)
#     # =====================================================
#     try:
#         A0 = max(y_fit[0], 1e-3)
#         B0 = 1 / (x_fit.max() - x_fit.min() + 1e-9)

#         popt, pcov = curve_fit(
#             exponential_model,
#             x_fit,
#             y_fit,
#             p0=[A0, B0],
#             sigma=sigma,
#             absolute_sigma=True,
#             bounds=([0, -np.inf], [np.inf, np.inf]),
#             maxfev=10000
#         )

#         A_exp, B_exp = popt
#         A_err, B_err = np.sqrt(np.diag(pcov))

#         y_exp = exponential_model(x_fit, A_exp, B_exp)
#         chi_exp = reduced_chi_square(y_fit, y_exp, sigma, 2)

#         results["exponential"] = {
#             "A": A_exp,
#             "A_err": A_err,
#             "B": B_exp,
#             "B_err": B_err,
#             "chi2_red": chi_exp
#         }

#     except Exception as e:
#         print(f"Exponential fit failed for {label}: {e}")
#         results["exponential"] = {
#             "A": np.nan,
#             "A_err": np.nan,
#             "B": np.nan,
#             "B_err": np.nan,
#             "chi2_red": np.nan
#         }

#     # =====================================================
#     # Print summary
#     # =====================================================
#     print(f"\n--- {label} ---")

#     print(
#         f"Constant fit: "
#         f"C = {C:.3f} ± {C_err:.3f}, "
#         f"χ²_red = {chi_const:.2f}"
#     )

#     print(
#         f"Linear fit: "
#         f"a = {a_lin:.4f} ± {a_lin_err:.4f}, "
#         f"b = {b_lin:.3f} ± {b_lin_err:.3f}, "
#         f"χ²_red = {chi_lin:.2f}"
#     )

#     print(
#         f"Exponential fit: "
#         f"A = {results['exponential']['A']:.3f} ± {results['exponential']['A_err']:.3f}, "
#         f"B = {results['exponential']['B']:.5f} ± {results['exponential']['B_err']:.5f}, "
#         f"χ²_red = {results['exponential']['chi2_red']:.2f}"
#     )

#     results["_x_fit"] = x_fit
#     results["_y_fit"] = y_fit
#     results["_sigma"] = sigma

#     return results


def hist_data(dt, bins):
    dt = dt.dropna()
    counts, edges = np.histogram(dt, bins=bins)
    centers = 0.5 * (edges[:-1] + edges[1:])
    return centers, counts


# ---------------------------
# Load Data
# ---------------------------
df_rand_train = load_random("Train")
df_rand_val   = load_random("Val")

# These should contain all models, with a 'model_name' column
df_model_train_all = load_model("Train")
df_model_val_all   = load_model("Val")

train_models = df_model_train_all["model_name"].unique() if not df_model_train_all.empty else []
val_models   = df_model_val_all["model_name"].unique() if not df_model_val_all.empty else []
all_models   = sorted(set(train_models).union(set(val_models)))

print("\nModels found:", all_models)

# Example
# results_random = compute_fits_with_uncertainties(x_rand, y_rand, "Random validation")
# results_modelA = compute_fits_with_uncertainties(x_modelA, y_modelA, "Model A validation")
# results_modelInA = compute_fits_with_uncertainties(x_modelInA, y_modelInA, "Model In-A validation")


## 11. Produce  model-vs-random lead-time plots

This final plotting section creates thesis-quality comparison figures for each model.

For every model, it:

1. selects validation anomalies for that model;
2. compares them with the validation random/control lead-time sample;
3. builds 1-hour lead-time bins;
4. fits constant, linear, and exponential functions to both distributions;
5. plots random/control and model counts with Poisson error bars;
6. adds fitted curves and reduced chi-square values;




In [ ]:
def plot_dt_distribution_thesis(x_r, y_r, res_r,
                                x_m, y_m, res_m,
                                N_rand, N_model,
                                title, ylim=(0, 70)):

    plt.rcParams['font.family'] = 'Cambria'
    plt.figure(figsize=(7, 5), dpi=600)

    # ---------- Random ----------
    if len(x_r) > 0:
        sigma_r = np.sqrt(y_r)
        plt.errorbar(
            x_r, y_r, yerr=sigma_r,
            fmt='o', markersize=4,
            color='tab:blue',
            capsize=3, alpha=0.7,
            label=f"Random (N={N_rand})"
        )

        if "exponential" in res_r and not np.isnan(res_r["exponential"]["A"]):
            A, B = res_r["exponential"]["A"], res_r["exponential"]["B"]
            x_fit = res_r["_x_fit"]
            plt.plot(
                x_fit, exponential_model(x_fit, A, B),
                color='tab:blue', linewidth=2
            )

    # ---------- Model ----------
    if len(x_m) > 0:
        sigma_m = np.sqrt(y_m)
        plt.errorbar(
            x_m, y_m, yerr=sigma_m,
            fmt='s', markersize=4,
            color='darkorange',
            capsize=3, alpha=0.7,
            label=f"Model (N={N_model})"
        )

        if "exponential" in res_m and not np.isnan(res_m["exponential"]["A"]):
            A, B = res_m["exponential"]["A"], res_m["exponential"]["B"]
            x_fit = res_m["_x_fit"]
            plt.plot(
                x_fit, exponential_model(x_fit, A, B),
                color='darkorange', linewidth=2
            )

    # ---------- Formatting ----------
    plt.xlabel(r"Lead time $\Delta t$ (hours)", fontsize=15)
    plt.ylabel("Event count", fontsize=15)
    plt.title(title, fontsize=17, fontweight="bold")

    plt.ylim(*ylim)
    plt.grid(axis="y", linestyle="--", alpha=0.4)
    plt.tick_params(axis='both', labelsize=13)

    plt.legend(frameon=False, fontsize=12)
    plt.tight_layout()
    plt.show()


def plot_dt_distribution_thesis_full(
    x_r, y_r, res_r,
    x_m, y_m, res_m,
    N_rand, N_model,model_short,
    title,
    ylim=(1, 100)
):
    """
    Thesis-standard plot:
    - Random vs Model
    - Constant, Linear, Exponential fits
    - Reduced chi-square in legend
    - Lowest chi-square highlighted
    - Log-scale y-axis
    """

    plt.rcParams['font.family'] = 'Cambria'
    plt.figure(figsize=(7, 5), dpi=600)

    # ---------- Helper to find best fit ----------
    def best_fit(res):
        chi_vals = {
            k: v["chi2_red"]
            for k, v in res.items()
            if isinstance(v, dict) and "chi2_red" in v and not np.isnan(v["chi2_red"])
        }
        return min(chi_vals, key=chi_vals.get) if chi_vals else None

    # ---------- RANDOM ----------
    if len(x_r) > 0:
        sigma_r = np.sqrt(y_r)
        plt.errorbar(
            x_r, y_r, yerr=sigma_r,
            fmt='o', markersize=4,
            color='tab:blue',
            capsize=3, alpha=0.7,
            label=f"Random (N={N_rand})"
        )

        best_r = best_fit(res_r)

        # Constant
        if "constant" in res_r:
            C = res_r["constant"]["C"]
            chi = res_r["constant"]["chi2_red"]
            plt.plot(
                x_r, np.full_like(x_r, C),
                linestyle="--", color="tab:blue", linewidth=1,
                alpha=0.7,
                label=f"R const ($\chi^2_{{\mathrm{{red}}}}={chi:.2f}$)"
            )

        # Linear
        if "linear" in res_r:
            a, b = res_r["linear"]["a"], res_r["linear"]["b"]
            chi = res_r["linear"]["chi2_red"]
            plt.plot(
                x_r, a * x_r + b,
                linestyle="-.", color="tab:blue", linewidth=1,
                alpha=0.7,
                label=rf"R lin ($\chi^2_{{\mathrm{{red}}}}={chi:.2f}$)"

            )

        # Exponential
        if "exponential" in res_r and not np.isnan(res_r["exponential"]["A"]):
            A, B = res_r["exponential"]["A"], res_r["exponential"]["B"]
            chi = res_r["exponential"]["chi2_red"]
            plt.plot(
                x_r, exponential_model(x_r, A, B),
                linestyle="-", color="tab:blue", linewidth=2,
                label=f"R exp ($\chi^2_{{\mathrm{{red}}}}={chi:.2f}$)"
            )

    # ---------- MODEL ----------
    if len(x_m) > 0:
        sigma_m = np.sqrt(y_m)
        plt.errorbar(
            x_m, y_m, yerr=sigma_m,
            fmt='s', markersize=4,
            color='darkorange',
            capsize=3, alpha=0.7,
            label=f"Model (N={N_model})"
        )

        best_m = best_fit(res_m)

        # Constant
        if "constant" in res_m:
            C = res_m["constant"]["C"]
            chi = res_m["constant"]["chi2_red"]
            plt.plot(
                x_m, np.full_like(x_m, C),
                linestyle="--", color="darkorange", linewidth=1,
                alpha=0.7,
                label=rf"M const ($\chi^2_{{\mathrm{{red}}}}={chi:.2f}$)"

            )

        # Linear
        if "linear" in res_m:
            a, b = res_m["linear"]["a"], res_m["linear"]["b"]
            chi = res_m["linear"]["chi2_red"]
            plt.plot(
                x_m, a * x_m + b,
                linestyle="-.", color="darkorange", linewidth=1,
                alpha=0.7,
                label=rf"M lin ($\chi^2_{{\mathrm{{red}}}}={chi:.2f}$)"

            )

        # Exponential
        if "exponential" in res_m and not np.isnan(res_m["exponential"]["A"]):
            A, B = res_m["exponential"]["A"], res_m["exponential"]["B"]
            chi = res_m["exponential"]["chi2_red"]
            plt.plot(
                x_m, exponential_model(x_m, A, B),
                linestyle="-", color="darkorange", linewidth=2,
                label=f"M exp ($\chi^2_{{\mathrm{{red}}}}={chi:.2f}$)"
            )

    # ---------- Formatting ----------
    plt.xlabel(r"Lead time $\Delta t$ (hours)", fontsize=15)
    plt.ylabel("Event count", fontsize=15)
    plt.yscale("log")
    plt.ylim(*ylim)

    plt.title(title, fontsize=17, fontweight="bold")
    plt.grid(axis="y", linestyle="--", alpha=0.4)
    plt.tick_params(axis='both', labelsize=13)

    plt.legend(frameon=False, fontsize=10, ncol=2)
    plt.tight_layout()
    plt.savefig(f"C:\PROJECT-DEMETER\Paper1\Plots\SW22TW48\leadtime_{model_short}.png",dpi=600)
    plt.show()
fit_results = {}  
for model_name in all_models:
    print("\n" + "="*80)
    print(f"Processing model: {model_name}")
    fit_results[model_name] = {}

    model_short = model_name.split("_")[-1]

    # ===========================
    # TRAIN
    # ===========================
    df_m_val = df_model_val_all[df_model_val_all["model_name"] == model_name]
    dt_rand_val  = df_rand_val["delta_t_hours"].dropna() if "delta_t_hours" in df_rand_val.columns else pd.Series(dtype=float)
    dt_model_val = df_m_val["delta_t_hours"].dropna()        if "delta_t_hours" in df_m_val.columns else pd.Series(dtype=float)

    n_rand_val  = len(dt_rand_val)
    n_model_val = len(dt_model_val)

    if n_rand_val > 0 or n_model_val > 0:
        all_dt_val = pd.concat([dt_rand_val, dt_model_val]).dropna()
        if not all_dt_val.empty:
            min_val = np.floor(all_dt_val.min())
            max_val = np.ceil(all_dt_val.max())
            bins_val = np.arange(min_val, max_val + 2, 1)

            x_rv, y_rv = hist_data(dt_rand_val,  bins_val) if n_rand_val  > 0 else (np.array([]), np.array([]))
            x_mv, y_mv = hist_data(dt_model_val, bins_val) if n_model_val > 0 else (np.array([]), np.array([]))
            # Example
            # results_random = compute_fits_with_uncertainties(x_rand, y_rand, "Random validation")
            # results_modelA = compute_fits_with_uncertainties(x_modelA, y_modelA, "Model A validation")
            # results_modelInA = compute_fits_with_uncertainties(x_modelInA, y_modelInA, "Model In-A validation")

            res_rv = compute_fits(x_rv, y_rv, f"Random Val – {model_short}") if len(x_rv) > 0 else {}
            res_mv = compute_fits(x_mv, y_mv, f"Model Val – {model_short}")  if len(x_mv) > 0 else {}
            
            # res_rv = compute_fits_with_uncertainties(x_rv, y_rv, f"Random Val – {model_short}") if len(x_rv) > 0 else {}
            # res_mv = compute_fits_with_uncertainties(x_mv, y_mv, f"Model Val – {model_short}")  if len(x_mv) > 0 else {}


            fit_results[model_name]["Random Val"] = res_rv
            fit_results[model_name]["Model Val"]  = res_mv

            plot_dt_distribution_thesis_full(
    x_rv, y_rv, res_rv,
    x_mv, y_mv, res_mv,
    n_rand_val, n_model_val,model_short,
    title=f"Lead-time distribution (Validation) – {model_short}"
)


## Briefing: how random sampling is done in this notebook

The random/control sampling in this notebook is not created by shuffling earthquake times inside this notebook. Instead, it starts from already prepared random/control files named like:

- `RDM_df_train_30D-22SW-tw48_w<window>.csv`
- `RDM_df_val_30D-22SW-tw48_w<window>.csv`

The procedure used here is:

1. **Load one random/control file for a rolling window.**  
   The notebook loops over windows \(w = 0,\ldots,15\).

2. **Keep only rows with `label == 1`.**  
   This means the sampled rows are random/control anomaly rows that already satisfy the seismic matching condition under the SW22/TW48 rule.

4. **Extract the first matched earthquake.**  
   The `matched_eqs` column can contain multiple matched earthquakes. The helper function takes only the first listed earthquake time.

5. **Compute lead time.**  
   For each sampled row:
   \[
   \Delta t = t_{eq} - t_n
   \]
   where \(t_n\) is the anomaly time-window timestamp and \(t_{eq}\) is the first matched earthquake time. The result is stored as `delta_t_hours`.

6. **Save the sampled control table.**  
   The output is saved as `_Random100.csv`, although the current train/validation loop uses only 100 iterations per window. With 16 windows, the current code produces approximately \(16 \times 100 = 1600\) sampled rows per split, assuming every window has available `label == 1` rows.



### Important limitation

Because the notebook samples only `label == 1` rows from the random/control files, the resulting baseline is conditional on random/control anomalies that already matched an earthquake within SW22/TW48. Therefore, this analysis compares the **lead-time distribution among matched anomalies**, not the full probability of obtaining an earthquake match from all candidate windows.
